# 16 — Consistent Hashing Ring (Amazon FAR style)

Distribute keys across nodes so that adding/removing a node moves as few keys as possible (sharded
caches, distributed stores, load balancers). Parts 1-3 build the core (concurrency at Part 3); Parts
4-5 are stretch tasks (replication, then parallel assignment). Fill stubs, run each test cell, peek at
the collapsed `ref_` solutions only after trying.

We use a **stable** hash (`hashlib.md5`, not the salted built-in `hash`) so placement is reproducible
and matches across processes. `_h(s)` is provided.

---

## Part 1 — Build the ring & look up

With `vnodes` virtual nodes per physical node (for balance), implement:
- `build_ring(nodes, vnodes) -> list[(hash, node)]`: place `vnodes` points per node (hash of
  `f"{node}#{v}"`), sorted by hash.
- `get_node(ring, key) -> node`: the first node **clockwise** from `hash(key)` (wrap around at the
  end) — use binary search (`bisect`).

**Lock down:** virtual nodes smooth the distribution; lookup is O(log n); the ring wraps, so a key past
the last point maps to the first.

In [ ]:
import bisect
import hashlib


def _h(s):
    return int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16)


def build_ring(nodes, vnodes):
    raise NotImplementedError


def get_node(ring, key):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    from collections import Counter
    ring = build_ring(["a", "b", "c"], vnodes=50)
    assert len(ring) == 150
    assert ring == sorted(ring)                        # sorted by hash
    assert get_node(ring, "key1") == get_node(ring, "key1") and get_node(ring, "key1") in {"a", "b", "c"}
    dist = Counter(get_node(ring, "k%d" % i) for i in range(3000))
    assert set(dist) == {"a", "b", "c"} and all(v > 300 for v in dist.values())   # reasonably balanced
    print("Part 1 OK")

_t1()

## Part 2 — Minimal movement on rebalance

- `assign(ring, keys) -> dict[key, node]`.
- `keys_moved(old_ring, new_ring, keys) -> set`: keys whose owner changed between two rings.

Show the consistent-hashing property: adding one node to a 3-node ring moves only ~1/4 of keys (not
all of them), and every moved key now lands on the **new** node.

**Lock down:** this is the whole point of consistent hashing vs `hash(key) % N` (which reshuffles
*everything* when N changes).

In [ ]:
def assign(ring, keys):
    raise NotImplementedError


def keys_moved(old_ring, new_ring, keys):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    keys = ["k%d" % i for i in range(2000)]
    r3 = build_ring(["a", "b", "c"], 50)
    r4 = build_ring(["a", "b", "c", "d"], 50)
    moved = keys_moved(r3, r4, keys)
    assert 0 < len(moved) < len(keys) // 2             # only a fraction move, not everything
    new = assign(r4, keys)
    assert all(new[k] == "d" for k in moved)           # movers all land on the added node
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe ring

`ConcurrentRing(vnodes)` with `add_node(n)`, `remove_node(n)`, `get_node(key)` (None if empty),
thread-safe so lookups can run while membership changes. Adding/removing an already-(absent/present)
node is a no-op.

**The invariant:** under concurrent churn (threads adding/removing nodes) and concurrent lookups, no
lookup ever crashes or returns a removed node mid-mutation. **Discuss:** guarding the sorted ring +
membership set together; read/write lock if reads dominate; keeping `get_node` O(log n).

In [ ]:
import threading


class ConcurrentRing:
    def __init__(self, vnodes):
        raise NotImplementedError

    def add_node(self, n):
        raise NotImplementedError

    def remove_node(self, n):
        raise NotImplementedError

    def get_node(self, key):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    r = ConcurrentRing(vnodes=20)
    for n in ("a", "b", "c"):
        r.add_node(n)
    errors = []

    def reader():
        try:
            for i in range(2000):
                assert r.get_node("k%d" % i) in {"a", "b", "c", "d", "e", None}
        except Exception as e:        # noqa: BLE001
            errors.append(e)

    def churner():
        for _ in range(50):
            r.add_node("d"); r.remove_node("d"); r.add_node("e"); r.remove_node("e")

    ts = [threading.Thread(target=reader) for _ in range(4)] + \
         [threading.Thread(target=churner) for _ in range(2)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert not errors
    assert r.get_node("x") in {"a", "b", "c"}          # churned nodes all removed at the end
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Replication

`get_replicas(ring, key, n) -> list`: the first `n` **distinct** nodes clockwise from `hash(key)` (for
storing `n` replicas of a key). The first element is the primary (`== get_node`). If fewer than `n`
distinct nodes exist, return all of them.

**Discuss:** why you must skip duplicate physical nodes while walking virtual nodes; preference lists
(Dynamo-style); weighted nodes (more vnodes = more share).

In [ ]:
def get_replicas(ring, key, n):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    ring = build_ring(["a", "b", "c"], 50)
    reps = get_replicas(ring, "key1", 2)
    assert len(reps) == 2 and len(set(reps)) == 2 and reps[0] == get_node(ring, "key1")
    alln = get_replicas(ring, "key1", 10)               # more than nodes available
    assert set(alln) == {"a", "b", "c"} and len(alln) == 3
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel assignment

`parallel_assign(keys, ring, num_procs) -> dict[key, node]`: assign a large key set across processes
with `ProcessPoolExecutor`; worker is `ringhash_workers.assign_chunk`. Must match the sequential
`assign`.

**Discuss:** key assignment is independent per key (embarrassingly parallel); the stable hash is what
makes parent and workers agree; pickling the ring to each worker.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import ringhash_workers


def parallel_assign(keys, ring, num_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    ring = build_ring(["a", "b", "c"], 50)
    keys = ["k%d" % i for i in range(200)]
    assert parallel_assign(keys, ring, 4) == {k: get_node(ring, k) for k in keys}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Bounded-load consistent hashing; jump consistent hash; rendezvous (HRW) hashing.
- Rebalancing data movement when a node joins/leaves; hinted handoff.
- Weighted nodes and hot-key mitigation.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import bisect
import hashlib
import threading
from concurrent.futures import ProcessPoolExecutor
import ringhash_workers


def _h(s):
    return int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16)


def ref_build_ring(nodes, vnodes):
    ring = []
    for n in nodes:
        for v in range(vnodes):
            ring.append((_h("%s#%d" % (n, v)), n))
    ring.sort()
    return ring


def ref_get_node(ring, key):
    ks = [h for h, _ in ring]
    i = bisect.bisect(ks, _h(key)) % len(ring)
    return ring[i][1]


def ref_assign(ring, keys):
    return {k: ref_get_node(ring, k) for k in keys}


def ref_keys_moved(old_ring, new_ring, keys):
    a, b = ref_assign(old_ring, keys), ref_assign(new_ring, keys)
    return {k for k in keys if a[k] != b[k]}


class RefConcurrentRing:
    def __init__(self, vnodes):
        self.vnodes = vnodes
        self.ring = []
        self.nodes = set()
        self.lock = threading.Lock()

    def add_node(self, n):
        with self.lock:
            if n in self.nodes:
                return
            self.nodes.add(n)
            for v in range(self.vnodes):
                bisect.insort(self.ring, (_h("%s#%d" % (n, v)), n))

    def remove_node(self, n):
        with self.lock:
            if n not in self.nodes:
                return
            self.nodes.discard(n)
            self.ring = [(h, x) for h, x in self.ring if x != n]

    def get_node(self, key):
        with self.lock:
            if not self.ring:
                return None
            ks = [h for h, _ in self.ring]
            i = bisect.bisect(ks, _h(key)) % len(self.ring)
            return self.ring[i][1]


def ref_get_replicas(ring, key, n):
    if not ring:
        return []
    ks = [h for h, _ in ring]
    start = bisect.bisect(ks, _h(key)) % len(ring)
    res = []
    for j in range(len(ring)):
        node = ring[(start + j) % len(ring)][1]
        if node not in res:
            res.append(node)
            if len(res) == n:
                break
    return res


def ref_parallel_assign(keys, ring, num_procs):
    chunks = [keys[i::num_procs] for i in range(num_procs)]
    out = {}
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for d in ex.map(ringhash_workers.assign_chunk, [(c, ring) for c in chunks]):
            out.update(d)
    return out


_r = ref_build_ring(["a", "b"], 10)
assert len(_r) == 20 and ref_get_node(_r, "z") in {"a", "b"}
_r3 = ref_build_ring(["a", "b", "c"], 40); _r4 = ref_build_ring(["a", "b", "c", "d"], 40)
_ks = ["k%d" % i for i in range(500)]
_m = ref_keys_moved(_r3, _r4, _ks)
assert 0 < len(_m) < 500 and all(ref_assign(_r4, _ks)[k] == "d" for k in _m)
assert ref_get_replicas(_r, "z", 5) == list(dict.fromkeys(ref_get_replicas(_r, "z", 5)))
assert ref_parallel_assign(_ks, _r3, 4) == ref_assign(_r3, _ks)
print("reference solutions OK")